## Configuracion de esta corrida

In [ ]:
EXPS = [{'id': 'E2', 'cambio': 'imgsz 416', 'imgsz': 416}, {'id': 'E3', 'cambio': 'imgsz 320', 'imgsz': 320}]

## Funciones comunes (eclib)

In [ ]:
import os, json, time, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2

CLASS_NAMES = ["G-cocci", "G+cocci", "G-bacilli", "G+bacilli"]
GB, GPB = 2, 3          # ids de G-bacilli y G+bacilli
NMS_IOU = 0.7

def prepare_ds(src_root, dst_root="/kaggle/temp/yolo_ds", with_bg=False):
    """Copia el dataset a un lugar escribible y escribe el yaml con rutas reales."""
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not dst_root.exists(): shutil.copytree(src_root, dst_root)
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
    tr = "[images/train, images/bg_train]" if with_bg else "images/train"
    y = dst_root / ("data_bg.yaml" if with_bg else "data_base.yaml")
    y.write_text(f"path: {dst_root}\ntrain: {tr}\nval: images/val\ntest: images/test\nnames:\n{names}\n")
    return dst_root, y

def read_gt(label_path):
    a = np.loadtxt(label_path, ndmin=2) if Path(label_path).stat().st_size else np.zeros((0, 5))
    c = a[:, 0].astype(int); xc, yc, w, h = a[:, 1:5].T
    return c, np.stack([xc-w/2, yc-h/2, xc+w/2, yc+h/2], 1)

def predict_dets(model, img_paths, imgsz, conf=0.01, iou=NMS_IOU, batch=32, device=None, max_det=1000):
    """Detecciones de baja confianza; el barrido de umbral se hace despues sobre este cache."""
    out = {}
    paths = [str(p) for p in img_paths]
    for i in range(0, len(paths), batch):
        for p, r in zip(paths[i:i+batch], model.predict(paths[i:i+batch], imgsz=imgsz, conf=conf, iou=iou, verbose=False, device=device, max_det=max_det)):
            h, w = r.orig_shape
            b = r.boxes
            out[Path(p).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxy.cpu().numpy() / np.array([w, h, w, h]))
    return out

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    i = iw * ih; ua = ((a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]))[:, None] + ((b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1]))[None] - i
    return i / np.maximum(ua, 1e-9)

def load_gts(ds_root, split):
    return {p.stem: read_gt(p) for p in sorted((Path(ds_root) / "labels" / split).glob("*.txt"))}

def counts(dets, gts, conf, classes):
    """Conteos por imagen (gt, pred) de las clases dadas."""
    rows = []
    for k, (gc, _) in gts.items():
        c, s, _b = dets[k]
        rows.append((int(np.isin(gc, classes).sum()), int((np.isin(c, classes) & (s >= conf)).sum())))
    return np.array(rows)

def count_metrics(dets, gts, conf, classes):
    a = counts(dets, gts, conf, classes); e = a[:, 1] - a[:, 0]
    return dict(mae=float(np.abs(e).mean()), bias=float(e.mean()), exact=float((e == 0).mean()),
                gt_total=int(a[:, 0].sum()), pred_total=int(a[:, 1].sum()))

def boot_ci(dets, gts, conf, classes, n=1000, seed=0):
    a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0]); rng = np.random.default_rng(seed)
    m = [e[rng.integers(0, len(e), len(e))].mean() for _ in range(n)]
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def boot_ci_groups(dets, gts, conf, classes, group_of, n=1000, seed=0, stat="mae"):
    """IC 95 % remuestreando GRUPOS de origen (group_of: stem -> grupo), no imagenes."""
    keys = list(gts); a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0])
    g = np.array([group_of[k] for k in keys]); ug, inv = np.unique(g, return_inverse=True)
    sums = np.bincount(inv, weights=e, minlength=len(ug)); cnt = np.bincount(inv, minlength=len(ug)); rng = np.random.default_rng(seed)
    m = []
    for _ in range(n):
        ix = rng.integers(0, len(ug), len(ug)); m.append(sums[ix].sum() / cnt[ix].sum())
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def match_stats(dets, gts, conf, cls=GB, thr=0.5):
    """TP/FP/FN de la clase cls (IoU>=thr, 1 a 1 greedy) y confusiones con la otra clase de bacilos."""
    tp = fp = fn = conf_to_gpb = 0
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf
        pc, pb = c[m], b[m]
        g = gc == cls; gb_ = gbx[g]; p = pb[pc == cls]
        iou = box_iou(p, gb_); used_p, used_g = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(gb_))), reverse=True):
            if v < thr: break
            if i in used_p or j in used_g: continue
            used_p.add(i); used_g.add(j)
        t = len(used_p); tp += t; fp += len(p) - t; fn += len(gb_) - t
        other = GPB if cls == GB else GB
        un = [j for j in range(len(gb_)) if j not in used_g]
        if un:
            q = box_iou(pb[pc == other], gb_[un]); conf_to_gpb += int((q >= thr).any(0).sum()) if q.size else 0
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=2*P*R/max(P+R, 1e-9), gt_clase_predicha_como_otra=conf_to_gpb)

def sweep(dets, gts, classes, confs=np.round(np.arange(0.05, 0.81, 0.05), 2)):
    rows = []
    for c in confs:
        r = dict(conf=float(c), **count_metrics(dets, gts, c, classes))
        r.update({"f1_Gb": match_stats(dets, gts, c)["f1"]}); rows.append(r)
    return pd.DataFrame(rows)

def bg_fp(dets_bg, conf, classes=(GB,)):
    n = np.array([int((np.isin(c, classes) & (s >= conf)).sum()) for c, s, _ in dets_bg.values()])
    return dict(n_imgs=len(n), fp_por_imagen=float(n.mean()), imgs_con_fp=float((n > 0).mean()))

def cpu_latency_ms(model, img, imgsz, n=100, warm=10):
    for _ in range(warm): model.predict(img, imgsz=imgsz, device="cpu", verbose=False)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model.predict(img, imgsz=imgsz, device="cpu", verbose=False); t.append((time.perf_counter()-t0)*1000)
    return float(np.median(t))

def ap_table(m):
    """mAP50 y mAP50-95 por clase desde un objeto DetMetrics de ultralytics."""
    idx = list(m.box.ap_class_index)
    return {CLASS_NAMES[c]: dict(ap50=float(m.box.ap50[i]), ap=float(m.box.ap[i])) for i, c in enumerate(idx)}


# Entrenamiento GPU: lista de experimentos (EXPS inyectada), evaluacion en VAL con conteo
Test NO se toca aqui. Cada experimento: train -> val ultralytics -> cache de detecciones -> barrido de conf -> IC por grupos -> latencia CPU proxy.

## Entorno y datos

In [ ]:
import os, json, time, shutil, subprocess, pickle
from pathlib import Path
import numpy as np, pandas as pd
r_ = subprocess.run("pip install -q ultralytics", shell=True, capture_output=True, text=True); print("pip ultralytics rc", r_.returncode, r_.stderr[-400:])
import torch
pf = subprocess.run("pip freeze | grep -i -E '^(ultralytics|ultralytics-thop|torch|torchvision|tensorflow|onnx|onnx2tf|onnxruntime|numpy|tflite)'", shell=True, capture_output=True, text=True).stdout
Path("/kaggle/working").mkdir(exist_ok=True); (Path("/kaggle/working") / "pip_freeze.txt").write_text(pf); print(pf)
import ultralytics; print("ultralytics", ultralytics.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
from ultralytics import YOLO
WORK = Path("/kaggle/working"); OUT = WORK / "outputs"; OUT.mkdir(exist_ok=True)
SRC = next(Path("/kaggle/input").glob("**/yolo_ds")); print("datos:", SRC)
DSR = Path("/kaggle/temp/yolo_ds")
if not DSR.exists(): shutil.copytree(SRC, DSR)
names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
(DSR / "data_base.yaml").write_text(f"path: {DSR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n")
(DSR / "data_bg.yaml").write_text(f"path: {DSR}\ntrain: [images/train, images/bg_train]\nval: images/val\ntest: images/test\nnames:\n{names}\n")
man = pd.read_csv(DSR / "split_manifest.csv"); GROUP_OF = dict(zip(man.name.str.replace(".jpg", "", regex=False), man.group))
VAL_IMGS = sorted((DSR / "images" / "val").glob("*.jpg")); GTS = load_gts(DSR, "val")
print("val:", len(VAL_IMGS), "imagenes")
assert not list(DSR.glob("labels/test/*.cache")) or True

## Funcion de experimento

In [ ]:
def stamp(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

def run_exp(cfg):
    eid, imgsz, seed = cfg["id"], cfg["imgsz"], cfg.get("seed", 42)
    t0 = time.time()
    args = dict(data=str(DSR / ("data_bg.yaml" if cfg.get("bg") else "data_base.yaml")), epochs=cfg.get("epochs", 100), patience=cfg.get("patience", 25),
                batch=16, imgsz=imgsz, seed=seed, device=0, workers=cfg.get("workers", 3), flipud=0.5, fliplr=0.5, degrees=0.0,
                project=str(WORK / "runs"), name=eid, exist_ok=True, plots=False, max_det=1000, verbose=False)
    if "fraction" in cfg: args["fraction"] = cfg["fraction"]
    args.update(cfg.get("extra", {}))
    stamp("train start"); model = YOLO(cfg.get("weights", "yolov8n.pt")); model.train(**args); stamp("train done")
    t_train = time.time() - t0
    best = WORK / "runs" / eid / "weights" / "best.pt"; shutil.copy2(best, OUT / f"{eid}_best.pt")
    shutil.copy2(WORK / "runs" / eid / "results.csv", OUT / f"{eid}_results.csv")
    m = YOLO(str(best)); v = m.val(data=str(DSR / "data_base.yaml"), split="val", imgsz=imgsz, conf=0.001, iou=0.7, max_det=1000, device=0, verbose=False, plots=False)
    per = {}
    for k, c in enumerate(v.box.ap_class_index):
        per[CLASS_NAMES[c]] = dict(ap50=float(v.box.ap50[k]), p=float(v.box.p[k]), r=float(v.box.r[k]))
    stamp("val ultralytics done"); dets = predict_dets(m, VAL_IMGS, imgsz, conf=0.01, iou=NMS_IOU, device=0)
    pickle.dump(dets, open(OUT / f"{eid}_val_dets.pkl", "wb"))
    sw = {}
    for nm, cl in (("estricta", [GB]), ("amplia", [GB, GPB])):
        s = sweep(dets, GTS, cl); s.to_csv(OUT / f"{eid}_sweep_{nm}.csv", index=False); sw[nm] = s
    sg = sw["estricta"]; row_mae = sg.loc[sg.mae.idxmin()]; row_f1 = sg.loc[sg.f1_Gb.idxmax()]
    conf_c = float(row_mae.conf); ms = match_stats(dets, GTS, conf_c)
    ci = boot_ci_groups(dets, GTS, conf_c, [GB], GROUP_OF)
    stamp("sweep done"); lat = cpu_latency_ms(m, str(VAL_IMGS[0]), imgsz, n=30, warm=5); stamp("latency done")
    rec = dict(id=eid, cambio=cfg["cambio"], semilla=seed, imgsz=imgsz, epochs_pedidas=args["epochs"], epochs_hechas=int(len(pd.read_csv(OUT / f"{eid}_results.csv"))),
               mAP50_Gb=per.get("G-bacilli", {}).get("ap50"), mAP50_todas=float(v.box.map50), mAP50_95_todas=float(v.box.map),
               conf_mae_opt=conf_c, mae_Gb=float(row_mae.mae), mae_ci_lo=ci[0], mae_ci_hi=ci[1], bias=float(row_mae.bias),
               P_Gb=ms["precision"], R_Gb=ms["recall"], F1_Gb=ms["f1"], Gb_pred_como_Gpb=ms["gt_clase_predicha_como_otra"],
               conf_f1_opt=float(row_f1.conf), f1_opt=float(row_f1.f1_Gb), mae_amplia=float(sw["amplia"].mae.min()),
               lat_cpu_ms_pt=lat, params=int(sum(p.numel() for p in m.model.parameters())), min_train=round(t_train / 60, 1), pesos=f"{eid}_best.pt")
    (OUT / f"{eid}_metrics.json").write_text(json.dumps(dict(rec=rec, per_class=per), indent=1))
    pd.DataFrame([rec]).to_csv(OUT / "experiments.csv", mode="a", header=not (OUT / "experiments.csv").exists(), index=False)
    print(json.dumps(rec, indent=1)); print(json.dumps(per, indent=1))
    if cfg.get("export_test"):
        for tag, kw in (("onnx", "format='onnx'"), ("tflite_fp16", "format='tflite', half=True"), ("tflite_int8", f"format='tflite', int8=True, data='{DSR / 'data_base.yaml'}'")):
            code = f"from ultralytics import YOLO; print('EXPORT OK', YOLO('{best}').export(imgsz={imgsz}, {kw}))"
            t1 = time.time()
            try:
                r = subprocess.run(["python", "-c", code], capture_output=True, text=True, timeout=900)
                print(f"[{tag}] rc={r.returncode} {time.time()-t1:.0f}s", (r.stdout + r.stderr)[-700:], flush=True)
            except subprocess.TimeoutExpired:
                print(f"[{tag}] TIMEOUT 900s", flush=True)
    shutil.rmtree(WORK / "runs" / eid / "weights", ignore_errors=True) if not cfg.get("export_test") else None
    return rec

## Ejecutar experimentos

In [ ]:
status = {"done": [], "failed": []}
for cfg in EXPS:
    try:
        run_exp(cfg); status["done"].append(cfg["id"])
    except Exception as e:
        import traceback; status["failed"].append((cfg["id"], repr(e))); print(traceback.format_exc())
    (OUT / "status.json").write_text(json.dumps(status))
print(status)